# Stable Baselines - Hindsight Experience Replay on Highway Env

Github Repo: [https://github.com/DLR-RM/stable-baselines3](https://github.com/DLR-RM/stable-baselines3)

Highway env: [https://github.com/eleurent/highway-env](https://github.com/eleurent/highway-env)

Documentation is available online: [https://stable-baselines3.readthedocs.io/](https://stable-baselines3.readthedocs.io/)

## Install Dependencies and Stable Baselines Using Pip


```
pip install stable-baselines3[extra]
```

In [ ]:
# for autoformatting
# %load_ext jupyter_black

In [ ]:
# Install stable-baselines latest version
!pip install "stable-baselines3[extra]>=2.0.0a4"

In [ ]:
# Install highway-env
!pip install highway-env

## Import policy, RL agent, ...

In [13]:
import gymnasium as gym
import highway_env
import numpy as np

from stable_baselines3 import HerReplayBuffer, SAC, DDPG
from stable_baselines3.common.noise import NormalActionNoise

## Create the Gym env and instantiate the agent

For this example, we will be using the parking environment from the [highway-env](https://github.com/Farama-Foundation/HighwayEnv) repo by @eleurent.

The parking env is a goal-conditioned continuous control task, in which the vehicle must park in a given space with the appropriate heading.


![parking-env](https://raw.githubusercontent.com/eleurent/highway-env/gh-media/docs/media/parking-env.gif)



### Train Soft Actor-Critic (SAC) agent

Here, we use HER "future" goal sampling strategy, where we create 4 artificial transitions per real transition

Note: the hyperparameters (network architecture, discount factor, ...) were tuned for this task

In [14]:
env = gym.make("parking-v0")

In [3]:
# SAC hyperparams:
model = SAC(
    "MultiInputPolicy",
    env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy="future",
    ),
    verbose=1,
    buffer_size=int(1e6),
    learning_rate=1e-3,
    gamma=0.95,
    batch_size=256,
    policy_kwargs=dict(net_arch=[256, 256, 256]),
)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [6]:
# Train for 1e5 steps
model.learn(int(1e4))

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 87.5     |
|    ep_rew_mean     | -51.8    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 52       |
|    time_elapsed    | 6        |
|    total_timesteps | 350      |
| train/             |          |
|    actor_loss      | -2.41    |
|    critic_loss     | 0.0353   |
|    ent_coef        | 0.78     |
|    ent_coef_loss   | -0.832   |
|    learning_rate   | 0.001    |
|    n_updates       | 249      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 93.5     |
|    ep_rew_mean     | -54.8    |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 8        |
|    fps             | 47       |
|    time_elapsed    | 15       |
|    total_timesteps | 748      |
| train/             |          |
|    actor_los

KeyboardInterrupt: 

In [8]:

# Save the trained agent
model.save('her_sac_highway2')

RuntimeError: Unable to sample before the end of the first episode. We recommend choosing a value for learning_starts that is greater than the maximum number of timesteps in the environment.

In [4]:
# Load saved model
model = SAC.load('her_sac_highway', env=env)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


#### Evaluate the agent

In [12]:
# we use the gym >v.26 API here. Note that you could also wrap the env in a DummyVecEnv
# which allows you to use a simplified API
env = gym.make("parking-v0", render_mode="human")

obs, _ = env.reset()

# Evaluate the agent
episode_reward = 0
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    done = truncated or terminated
    episode_reward += reward
    if done or info.get("is_success", False):
        print("Reward:", episode_reward, "Success?", info.get("is_success", False))
        episode_reward = 0.0
        obs, _ = env.reset()

NameNotFound: Environment parking doesn't exist. Did you mean: `CarRacing`?

https://github.com/Farama-Foundation/HighwayEnv/blob/master/highway_env/envs/parking_env.py

https://highway-env.farama.org/rewards/

### Train parking-v0 agent with custom reward function

In [10]:
import gym
import numpy as np

class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env):
        super(CustomRewardWrapper, self).__init__(env)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)

        # Calculamos la recompensa personalizada
        custom_reward = self.compute_custom_reward(obs, reward, terminated, truncated, info)

        return obs, custom_reward, terminated, truncated, info

    def compute_custom_reward(self, obs, reward, terminated, truncated, info):
        """
        Calcula la recompensa personalizada considerando dos factores:
        1. Cercanía al objetivo usando una norma ponderada.
        2. Penalización por colisión si el vehículo ha chocado.
        """
        # Aseguramos que la observación sea una tupla
        obs = obs if isinstance(obs, tuple) else (obs,)

        # Calculamos la recompensa basada en la distancia al objetivo
        custom_reward = sum(
            self.compute_proximity_reward(
                agent_obs["achieved_goal"], agent_obs["desired_goal"]
            )
            for agent_obs in obs
        )

        # Obtenemos la instancia base del entorno
        base_env = self.get_base_env()

        # Penalización por colisiones
        collision_penalty = base_env.config["collision_reward"] # -5
        num_collisions = sum(v.crashed for v in base_env.controlled_vehicles)
        custom_reward += collision_penalty * num_collisions  # Restará puntos si hay colisión

        return custom_reward

    def compute_proximity_reward(self, achieved_goal, desired_goal):
        """
        Calcula la recompensa según qué tan cerca está el vehículo de su objetivo.
        Usa una métrica de distancia ponderada (norma Lp^p).
        """
        base_env = self.get_base_env()
        reward_weights = np.array(base_env.config["reward_weights"]) # [1, 0.3, 0, 0, 0.02, 0.02]

        # Calculamos la distancia ponderada entre la posición actual y la deseada
        distance = np.abs(achieved_goal - desired_goal)
        weighted_distance = np.dot(distance, reward_weights)

        # Usamos una potencia para ajustar la recompensa (p < 1 enfatiza pequeños errores)
        p = 0.5
        return -np.power(weighted_distance, p)

    def get_base_env(self):
        """
        Obtiene la instancia base del entorno, en caso de que haya varios *Wrappers* anidados.
        """
        base_env = self.env
        while hasattr(base_env, 'env'):
            base_env = base_env.env
        return base_env


In [15]:
# Create the original environment
env = gym.make("parking-v0")

# Wrap the environment with the custom reward wrapper
env = CustomRewardWrapper(env)

In [16]:
# SAC hyperparams:
model = SAC(
    "MultiInputPolicy",
    env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy="future",
    ),
    verbose=1,
    buffer_size=int(1e6),
    learning_rate=1e-3,
    gamma=0.95,
    batch_size=256,
    policy_kwargs=dict(net_arch=[256, 256, 256]),
)

Using cuda device


/home/ruben/w/CARLA_0.9.15/PythonAPI/.venv/lib/python3.10/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


NotImplementedError: Cannot convert space of type Dict('achieved_goal': Box(-inf, inf, (6,), float64), 'desired_goal': Box(-inf, inf, (6,), float64), 'observation': Box(-inf, inf, (6,), float64)). Please upgrade your code to gymnasium.

In [18]:
# Train for 1e5 steps
model.learn(int(1e4))

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 103      |
|    ep_rew_mean     | -34.6    |
|    success_rate    | 0.25     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 33       |
|    time_elapsed    | 12       |
|    total_timesteps | 413      |
| train/             |          |
|    actor_loss      | 2.96     |
|    critic_loss     | 0.00579  |
|    ent_coef        | 0.00431  |
|    ent_coef_loss   | 5.88     |
|    learning_rate   | 0.001    |
|    n_updates       | 15690    |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 131      |
|    ep_rew_mean     | -45.5    |
|    success_rate    | 0.25     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 31       |
|    time_elapsed    | 33       |
|    total_timesteps | 1047     |
| train/             |          |
|    actor_los

In [19]:
# we use the gym >v.26 API here. Note that you could also wrap the env in a DummyVecEnv
# which allows you to use a simplified API
env = gym.make("parking-v0", render_mode="human")
# Wrap the environment with the custom reward wrapper
env = CustomRewardWrapper(env)

obs, _ = env.reset()

# Evaluate the agent
episode_reward = 0
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    done = truncated or terminated
    episode_reward += reward
    if done or info.get("is_success", False):
        print("Reward:", episode_reward, "Success?", info.get("is_success", False))
        episode_reward = 0.0
        obs, _ = env.reset()

Reward: -14.57261375285167 Success? True
Reward: -94.04646686735316 Success? False
Reward: -9.242169251137026 Success? True
Reward: -25.680950354566903 Success? False
Reward: -8.604652225112334 Success? True
Reward: -6.763116718921042 Success? True
Reward: -12.187719115888763 Success? True
Reward: -8.070108377112485 Success? True
